n2p2 dataset might look like,

```python
begin                                       --- begining of one atomic configuration
lattice 5.0 0.0 0.0
lattice 0.0 5.0 0.0
lattice 0.0 0.0 5.0
atom 0.0 0.0 0.0 Si 0.0 0.0 0.0 0.0 0.0
atom 2.5 2.5 2.5 Si 0.0 0.0 0.0 0.0 0.0
energy -10.5
charge 0.0
end                                         --- end of one atomic configuration
```

Task -- getting atomic structures as ASE objects and energy values separately

In [ ]:
from pathlib import Path # class 'Path' that helps locating the files
from ase import Atoms # class 'Atoms' represents atomic structures
import numpy as np
from dscribe.descriptors import ACSF # library for generating descriptors of atomic structures

'Atom Object' is like,

```python
atoms = Atoms(
    symbols=["Fe", "Fe"],
    positions=[[0, 0, 0], [2, 2, 2]]
)
```

They can store elements, coordinates, unit cell dimensions, periodic boundry conditions

# TASK 1 - Extract data from the data file 

Internal Helper Functions 

+ not called directly by the user
+ supports the main function

_ at the beginning indicates that this is intended to be an internal helper function

In [ ]:
def _parse_n2p2_block(block): # Internal Helper function to parse a single n2p2 block and extract atomic structure and energy information from the lines in n2p2 file
    symbols = []                # block - lines belongs to one atomic configuration
    positions = []
    cell = []
    energy = None
    has_cell = False

    for line in block:
        parts = line.split() # separate the strings in a line based on whitespace

        if not parts:
            continue # if the list parts is empty, skip to next line

        if parts[0] == "lattice":
            cell.append([float(x) for x in parts[1:4]]) # Make the vector with direction values of [x,y,z] ex - from "lattice 0.0 5.0 0.0" extract [0.0, 5.0, 0.0]
            has_cell = True # lattice information for this block is found successfully

        elif parts[0] == "atom":
            positions.append([float(x) for x in parts[1:4]]) # poistion of the atom to the list positions, ex - from "atom 0.0 0.0 0.0 H" extract [0.0, 0.0, 0.0]
            symbols.append(parts[4]) # Type of atom to the list symbols, ex - from "atom 0.0 0.0 0.0 Si" extract "Si"
                                       # ignores all the other information in the line, like charge, mass, etc. 

        elif parts[0] == "energy":
            energy = float(parts[1]) # extract the reference energy value

    if energy is None:
        raise ValueError("Energy missing in n2p2 block") # raising an error if energy is not found in the block

    if not symbols:
        raise ValueError("No atoms found in n2p2 block") # raising an error if no atoms are found in the block

    atoms = Atoms( #Converst extracted information into ASE Atoms object
        symbols=symbols,
        positions=positions,
        cell=cell if has_cell else None,
        pbc=has_cell # if TRUE, ASE treats the structure as periodically repeated in all three spatial directions.
    )

    return atoms, energy

Number of structures: 51702
Reference energies: [-1412.31194807 -1415.37624938 -1413.92285234 ... -1438.08844783
 -1444.23706662 -1449.31426814]


A 'block' looks like this
 
```python
block = [
    "lattice 5.0 0.0 0.0",
    "lattice 0.0 5.0 0.0",
    "lattice 0.0 0.0 5.0",
    "atom 0.0 0.0 0.0 Si 0.0 0.0 0.0 0.0 0.0",
    "energy -10.5"
]
```

<img src="images/cell example.png" width="300">

In [ ]:
def read_n2p2(path): # function to read the entire dataset from n2p2 file
    atoms_list = [] # store ASE atomic structures
    energies = []
    block = [] # temporarily stores the lines of one configuration
    inside_block = False # tracks whether Python is currently reading a configuration

    with open(path, "r") as file: # 'as file' assignss the opened file to the variable 'file' which can be used to read the file 
        for line in file:
            line = line.strip() #.strip() method removes whitespace at the beginning and end of each line, including newline characters

            if line == "begin": # when encounters 'begin', it knows a new atomic configuration is starting
                block = []
                inside_block = True # program is now reading a configuration

            elif line == "end":
                if not inside_block:
                    raise ValueError("Unexpected end")

                atoms, energy = _parse_n2p2_block(block) # now block contains the lines of one configuration, so it can be parsed to extract atomic structure and energy information
                atoms_list.append(atoms)
                energies.append(energy)
                inside_block = False #current configuration has completed,no longer reading a configuration

            elif inside_block:
                block.append(line) # if inside a configuration, add the line to the current block

    if inside_block:
        raise ValueError("Unclosed n2p2 block") # checks whether a configuration was started but never closed

    return atoms_list, np.array(energies) #returns the complete list of atomic structures and an array of reference energies
                                            #convert energie to numpy array for easier manipulation and analysis



# Example usage generated using AI
if __name__ == "__main__":

    path = Path.cwd() / "input.data"

    atoms_list, energies = read_n2p2(path)

    print("Number of structures:", len(atoms_list))
    print("Reference energies:", energies)

# TASK 2 - Building descriptors

Task - convert those atomic structures into numerical features using Atom-Centered Symmetry Functions (ACSF)

Descriptor - numerical representation that describes the local atomic environment, including the chemical species, spatial arrangement of neighboring atoms

ACSF

- local environment surrounding an atom using mathematical functions based on distances and angles

- provides features that are designed to be invariant to translation and rotation, and to permutations of equivalent atoms

- rotating or translating a structure should not change its potential energy


### Gaussian Radial Symmetry Function

A typical expression is:

$$
G_i^2 = \sum_{j \neq i} e^{-\eta(R_{ij}-R_s)^2} f_c(R_{ij})
$$

#### Where:

| Symbol | Meaning |
|--------|---------|
| $i$ | Central atom |
| $j$ | Neighboring atom |
| $R_{ij}$ | Distance between atoms $i$ and $j$ |
| $\eta$ | Controls the width of the Gaussian function |
| $R_s$ | Shifts the center of the Gaussian |
| $f_c$ | Cutoff function |


### Gaussian Angular Symmetry Function

consider a central atom \(i\) and two neighboring atoms \(j\) and \(k\), The angular descriptor captures information about the angle θ between neighboring atoms.

<img src="images/Angular symmetry functions.png" width="300">

### Averaging atomic descriptors


Consider the following matrix:

###### Example Atomic Descriptor Matrix

| Atom | Feature 1 | Feature 2 | Feature 3 |
|------|-----------|-----------|-----------|
| Atom 1 | 2 | 4 | 6 |
| Atom 2 | 4 | 6 | 8 |
| Atom 3 | 6 | 8 | 10 |
| **Average** | **4** | **6** | **8** |

`axis=0` calculates the mean down each column, across all atoms.


```python
np.mean(atomic_descriptors, axis=0)
```

**Result:**

```python
[4.0, 6.0, 8.0]
```

This is now one feature vector representing the entire structure.


Mathematically:

$$
\overline{G}_k = \frac{1}{N}\sum_{i=1}^{N}G_{ik}
$$

Where:

- $N$ = Number of atoms in the structure.
- $G_{ik}$ = Descriptor feature $k$ for atom $i$.
- $\overline{G}_k$ = Average descriptor feature $k$.

#### WhyAverage?

Different atomic structures may contain different numbers of atoms.

For example, one structure might contain 10 atoms and another 20 atoms.

Averaging converts both into fixed length vectors, making them suitable for a standard neural network.

limitation: averaging loses some information about individual atomic environments. In addition, it does not explicitly account for the number of atoms, which matters when predicting total energy rather than energy per atom.

this method is convenient for a simple task, but it is not equivalent to a full atom centered MLIP.


In [ ]:
# Generate one feature vector for each atomic configuration

def build_descriptors(atoms_list): # in atoms_list each element is an ASE Atoms object representing a single atomic configuration

    species = sorted({ # Python set is created using curley brackets , to automatically remove duplicates
        symbol          # Sorting is done to ensure consistent ordering of species across different configurations, the same order will be used again for traning and etc.
        for atoms in atoms_list
        for symbol in atoms.get_chemical_symbols() # ASE method - returns the chemical symbols of every atom in a structure , return would be smth like ["Fe", "Ni"]
    })

# ACSF descriptor generator

    acsf = ACSF( 
        species=species, # tells which elements exist in dataset
        r_cut=6.0, # cutoff radius - only the atoms inside this radius contribute to the descriptor

        # radial symmetry functions - how neighboring atoms are distributed at different distances from a central atom
        g2_params=[
            [1.0, 0.0], # [neta, Rs] - neta controls the width of the Gaussian function, Rs shifts the center of the Gaussian function (above equation)
            [1.0, 2.0],
            [1.0, 4.0] # 3 Gaussian functions allow the descriptor to distinguish between atomic environments with different neighbor distances.
        ],

        # angular symmetry functions - describe the relative arrangement of three atoms
        g4_params=[
            [1.0, 1.0, 1.0], # [neta, zeta, lambda]
            [1.0, 1.0, -1.0] # neta - Controls the distance dependent exponential term, zeta - Controls angular resolution, lambda - Controls the angular preference
        ] # here lambda is the only difference - distinguish atomic arrangements that may have similar radial distances but different bond angles.
    )

    features = [] # to store the descriptor vector for each atomic structure
                    # this will look like features = [
                    #                                    [0.2, 0.5, 0.8], # features of one atomic structure per inner list
                    #                                    [0.3, 0.6, 0.7],
                    #                                    [0.1, 0.4, 0.9]  ]

    for atoms in atoms_list:

        atomic_descriptors = acsf.create(atoms) # calculates the ACSF descriptor for every atom in the current structure

    # if contains four atoms and the descriptor generator produces six features per atom, ex :

      #  atomic_descriptors = [
       #          [0.2, 0.4, 0.7, 0.1, 0.3, 0.5],
      #          [0.3, 0.5, 0.8, 0.2, 0.4, 0.6],
       #         [0.4, 0.6, 0.9, 0.3, 0.5, 0.7],
      #          [0.5, 0.7, 1.0, 0.4, 0.6, 0.8]
       #                                         ]
    
        structure_descriptor = np.mean(         # neural network expects one fixed length input vector per atomic structure, not one descriptor vector per atom
        atomic_descriptors, axis=0              # calculate arithmetic average for one feature of all atoms, then get a single value
                                                # end up with a vector of length equal to the number of features per atom, that represets the entire atomic structure
                        )


        features.append(structure_descriptor) # adds the current structure's descriptor to the list of features for all structures



    # ex : after 3 iterations,

    # features = [
    #                [0.2, 0.4, 0.6],
    #               [0.3, 0.5, 0.7],        each row corresponds to one atomic structure, each column corresponds to one feature
    #                [0.4, 0.6, 0.8]
    #                                ] 

    
    X = np.array(features) # convert the list of features to a numpy array for easier manipulation and analysis

    return X


# Example usage generated using AI

if __name__ == "__main__":
    atoms_list, energies = read_n2p2("input.data")

    X = build_descriptors(atoms_list)

    print("Feature matrix shape:", X.shape)


KeyboardInterrupt: 